# Edits

Every change to a graph, from the browser or from a script, is an edit: a small record that the rules either apply or refuse. This notebook builds a graph edit by edit, shows the graph after each one, and then feeds the rules edits they must refuse, to show the reason each one gets.

In [1]:
# Standard library
import json

In [2]:
# Local libraries
import element_51
from element_51.edits import (
    MAX_CARD_SIZE,
    AddLink,
    AddNode,
    EditError,
    ResizeNode,
    SetValue,
)
from element_51.model import PortRef

In [3]:
# One number node, one text node and one that doubles a number
TOO_WIDE = MAX_CARD_SIZE + 1.0
CARD_WIDTH = 300.0
SHOWN = 3.5

## Node types to edit with

In [4]:
@element_51.node
def number() -> float:
    return SHOWN


@element_51.node
def double(value: float) -> float:
    return value * 2


@element_51.node
def shout(text: str) -> str:
    return text.upper()


app = element_51.App([number, double, shout])
print([registered.node_type.name for registered in (number, double, shout)])

['number', 'double', 'shout']


## An edit is a record

`AddNode` places a node with an id, a type name and a position. It is a pydantic model, so it has a JSON form, which is what the browser sends.

In [5]:
add_first = AddNode(id="n1", node_type="number", x=60.0, y=80.0)
print(add_first.model_dump_json())
assert json.loads(add_first.model_dump_json())["kind"] == "add_node"

{"kind":"add_node","id":"n1","node_type":"number","x":60.0,"y":80.0}


## Apply edits, one at a time

In [6]:
app.apply(add_first)
app.apply(AddNode(id="n2", node_type="double", x=320.0, y=80.0))
app.apply(SetValue(node="n2", port="value", value=SHOWN))
for placed in app.graph.nodes:
    print(placed.id, placed.node_type, (placed.x, placed.y), placed.values)
assert [placed.id for placed in app.graph.nodes] == ["n1", "n2"]
assert app.graph.nodes[1].values == {"value": SHOWN}

n1 number (60.0, 80.0) {}
n2 double (320.0, 80.0) {'value': 3.5}


A value on `n2.value` was set before linking. Taking it away with `None` frees the input for a link.

In [7]:
app.apply(SetValue(node="n2", port="value", value=None))
link = AddLink(
    source=PortRef(node="n1", port="out"),
    target=PortRef(node="n2", port="value"),
)
app.apply(link)
print(app.graph.links)
assert len(app.graph.links) == 1

(Link(source=PortRef(node='n1', port='out'), target=PortRef(node='n2', port='value')),)


## Resize a card

`ResizeNode` gives a card a width (and a height for the picture or chart on it). `None` means the default size.

In [8]:
app.apply(ResizeNode(id="n1", width=CARD_WIDTH, display_height=None))
print(app.graph.nodes[0].width, app.graph.nodes[0].display_height)
assert app.graph.nodes[0].width == CARD_WIDTH
app.apply(ResizeNode(id="n1", width=None, display_height=None))
assert app.graph.nodes[0].width is None

300.0 None


## Edits the rules refuse

A refused edit raises `EditError` with a kind and a sentence for the person who sees it, and leaves the graph as it was.

In [9]:
refused = {
    "a second node with the same id": AddNode(
        id="n1", node_type="number", x=0.0, y=0.0
    ),
    "a node type nobody registered": AddNode(
        id="n9", node_type="missing", x=0.0, y=0.0
    ),
    "text in a float input": SetValue(node="n2", port="value", value="text"),
    "a card wider than the limit": ResizeNode(
        id="n1", width=TOO_WIDE, display_height=None
    ),
}
before = app.graph
for what, edit in refused.items():
    try:
        app.apply(edit)
    except EditError as error:
        print(f"{what}: {error.kind.value}: {error}")
    else:
        msg = f"{what} was applied"
        raise AssertionError(msg)
assert app.graph == before

a second node with the same id: id_in_use: node id n1 is already in use
a node type nobody registered: unknown_node_type: unknown node type 'missing'
text in a float input: bad_value: 'text' is not a float for n2.value
a card wider than the limit: bad_size: width must be above 0 and at most 4000


A link between ports of different types is refused too.

In [10]:
app.apply(AddNode(id="n3", node_type="shout", x=320.0, y=240.0))
mismatch = AddLink(
    source=PortRef(node="n1", port="out"),
    target=PortRef(node="n3", port="text"),
)
refusal = None
try:
    app.apply(mismatch)
except EditError as error:
    refusal = error
assert refusal is not None
print(refusal.kind.value, refusal)
assert refusal.kind.value == "type_mismatch"

type_mismatch cannot link float to str
